In [1]:
!pip install pillow_heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 3.9 MB/s  0:00:01


In [2]:
# ===================================================================
# === 1. IMPORT LIBRARY ===
# ===================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import pillow_heif # Tambahkan library ini untuk membaca HEIC
import os
import pandas as pd
import numpy as np
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, accuracy_score
import cv2
import timm
from pathlib import Path
from tqdm.auto import tqdm

# Daftarkan opener HEIC ke Pillow
pillow_heif.register_heif_opener()

# ===================================================================
# === 2. KONFIGURASI DAN SETUP ===
# ===================================================================
# Direktori (Sesuaikan path ini dengan lokasi asli folder Anda)
TRAIN_DIR = "/kaggle/input/datasets/mdzkhkm/transdig/Dataset Transdik" 

# Konfigurasi Model dan Training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224 # Sesuaikan ukuran input gambar untuk model EVA-02 Base
BATCH_SIZE = 8
NUM_WORKERS = os.cpu_count()

# --- HYPERPARAMETER UNTUK GRADUAL UNFREEZING ---
EPOCHS_S1 = 3
LR_S1 = 3e-4
EPOCHS_S2 = 5
LR_S2 = 5e-5
EPOCHS_S3 = 15
LR_S3 = 1.59e-06 

# Statistik ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"Menggunakan device: {DEVICE}")
print(f"Ukuran gambar: {IMG_SIZE}x{IMG_SIZE}, Batch Size: {BATCH_SIZE}")

# ===================================================================
# === 3. FUNGSI DAN KELAS HELPER ===
# ===================================================================
def convert_path_to_df(dataset):
    image_dir = Path(dataset)
    
    # 1. Tentukan ekstensi gambar yang valid (termasuk .heic)
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff', '.heic'}

    # 2. Ambil semua file path terlebih dahulu
    all_filepaths = list(image_dir.glob(r'**/*.*'))
    
    # 3. Filter daftar file, hanya ambil yang ekstensinya ada di set
    filepaths = [p for p in all_filepaths if p.suffix.lower() in image_extensions]
    
    # Cek jika tidak ada gambar yang ditemukan
    if not filepaths:
        print(f"Peringatan: Tidak ada file gambar yang ditemukan di {image_dir}")
        return pd.DataFrame(columns=['Filepath', 'Label'])

    # 4. Buat DataFrame
    labels = [p.parts[-2] for p in filepaths]
    filepaths = pd.Series(filepaths, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')
    
    return pd.concat([filepaths, labels], axis=1)

class CustomDataset(Dataset):
    def __init__(self, dataframe, image_column, label_column=None, transform=None):
        self.dataframe = dataframe
        self.image_column = image_column
        self.label_column = label_column
        self.transform = transform
        
    def __len__(self): 
        return len(self.dataframe)
        
    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx][self.image_column]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform: 
            image = self.transform(image)
            
        if self.label_column:
            label = self.dataframe.iloc[idx][self.label_column]
            return image, torch.tensor(label, dtype=torch.long)
            
        return image

class DualTransformDataset(Dataset):
    def __init__(self, dataframe, image_column, label_column, transform_main, transform_extra):
        self.dataframe = dataframe
        self.image_column = image_column
        self.label_column = label_column
        self.transform_main = transform_main
        self.transform_extra = transform_extra

    def __len__(self):
        # Gandakan ukuran dataset (dua versi per gambar)
        return len(self.dataframe) * 2

    def __getitem__(self, idx):
        # Tentukan apakah pakai augmentasi utama atau tambahan
        base_idx = idx // 2
        use_extra = idx % 2 == 1

        row = self.dataframe.iloc[base_idx]
        img_path = row[self.image_column]
        label = torch.tensor(row[self.label_column], dtype=torch.long)

        image = Image.open(img_path).convert('RGB')

        if use_extra:
            image = self.transform_extra(image)
        else:
            image = self.transform_main(image)

        return image, label

class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        
    def __call__(self, img):
        img_np = np.array(img)
        img_lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2Lab)
        l, a, b = cv2.split(img_lab)
        l_clahe = self.clahe.apply(l)
        img_lab_clahe = cv2.merge((l_clahe, a, b))
        img_rgb_clahe = cv2.cvtColor(img_lab_clahe, cv2.COLOR_Lab2RGB)
        return Image.fromarray(img_rgb_clahe)

# ===================================================================
# === 4. PERSIAPAN DATA DAN SPLITTING TETAP ===
# ===================================================================
print("\n" + "="*50)
print("Memuat dan mempersiapkan data training...")

# --- Daftar file yang ingin di-exclude ---
files_to_exclude = { }

# --- Langkah 4.1: Memuat data training ---
train_df = convert_path_to_df(TRAIN_DIR)
print(f"Total gambar ditemukan sebelum filter: {len(train_df)}")

# --- FILTER DATASET: Membuang gambar yang ada di files_to_exclude ---
# Kita ambil nama filenya saja menggunakan Path(x).name, lalu cek apakah ada di set
train_df = train_df[~train_df['Filepath'].apply(lambda x: Path(x).name).isin(files_to_exclude)]
train_df = train_df.reset_index(drop=True)

print(f"Total gambar setelah file bermasalah dihapus: {len(train_df)}")

# --- Langkah 4.2: Pemetaan Label ---
label_mapping = {
    "Aljabbar": 0,       
    "Asrama": 1,         
    "GKU 1": 2,          
    "GKU 2": 3,          
    "GKU3": 4,           
    "GSG": 5,            
    "KOICA": 6,          
    "Labtek Baru": 7,    
    "Rektorat": 8,       
    "Transformer": 9     
}

train_df['Label'] = train_df['Label'].map(label_mapping)

# Drop baris yang labelnya tidak ada di mapping (untuk keamanan)
train_df = train_df.dropna(subset=['Label'])
train_df['Label'] = train_df['Label'].astype(int)

# --- Langkah 4.3: Split Data ---
train_split, val_split = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_df['Label']
)

# Reset index agar rapi
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

print(f"Ukuran set training: {len(train_split)}")
print(f"Ukuran set validasi: {len(val_split)}")
print("="*50 + "\n")

# --- Langkah 4.4: Augmentasi dan DataLoaders ---
train_transform = transforms.Compose([
    CLAHETransform(),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD)
])

# Transform tambahan: rotasi 90° kiri/kanan
train_transform_extra = transforms.Compose([
    CLAHETransform(),
    transforms.RandomChoice([
        transforms.RandomRotation((90, 90)),
        transforms.RandomRotation((-90, -90))
    ]),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transform = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = DualTransformDataset(
    train_split,
    image_column='Filepath',
    label_column='Label',
    transform_main=train_transform,
    transform_extra=train_transform_extra
)

val_dataset = CustomDataset(val_split, 'Filepath', 'Label', val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True)

# Hitung Class Weights untuk menangani imbalance data
class_weights = compute_class_weight("balanced", classes=np.unique(train_split["Label"]), y=train_split["Label"])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

Menggunakan device: cuda
Ukuran gambar: 224x224, Batch Size: 8

Memuat dan mempersiapkan data training...
Total gambar ditemukan sebelum filter: 323
Total gambar setelah file bermasalah dihapus: 323
Ukuran set training: 258
Ukuran set validasi: 65



In [3]:
# ===================================================================
# === 5. DEFINISI MODEL ===
# ===================================================================
class SingleEVAModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Cukup gunakan nama dasar arsitektur tanpa tag tambahan
        self.backbone = timm.create_model(
            "eva02_base_patch14_224", 
            pretrained=True, 
            num_classes=num_classes 
        )
    def forward(self, x):
        return self.backbone(x)

model = SingleEVAModel(num_classes=10).to(DEVICE)
print("Model EVA-02 berhasil dimuat!")


model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

Model EVA-02 berhasil dimuat!


In [4]:
# ===================================================================
# === 6. FUNGSI UNFREEZING & TRAINING DENGAN SAVE MODEL ===
# ===================================================================
def freeze_all(m):
    for param in m.parameters():
        param.requires_grad = False

def unfreeze_all(m):
    for param in m.parameters():
        param.requires_grad = True

def set_trainable_stage(model, stage):
    if stage == 1:
        # Stage 1: Freeze semua, hanya train classification head
        freeze_all(model)
        for param in model.backbone.head.parameters():
            param.requires_grad = True
        print("Mempersiapkan Stage 1: Hanya Classification Head yang dilatih.")
        
    elif stage == 2:
        # Stage 2: Unfreeze sebagian besar layer atas (6 block terakhir)
        freeze_all(model)
        for param in model.backbone.blocks[6:].parameters():
            param.requires_grad = True
        for param in model.backbone.norm.parameters():
            param.requires_grad = True
        for param in model.backbone.head.parameters():
            param.requires_grad = True
        print("Mempersiapkan Stage 2: Top-half dari backbone dan Head dilatih.")
        
    elif stage == 3:
        # Stage 3: Unfreeze seluruh model
        unfreeze_all(model)
        print("Mempersiapkan Stage 3: Seluruh layer model dilatih (Full Fine-tuning).")

def train_model(model, train_loader, val_loader, optimizer, num_epochs, stage_name, save_best=False, model_save_path="best_model.pth"):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val_acc = 0.0 # Variabel untuk melacak skor terbaik
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        # Training Phase
        for images, labels in tqdm(train_loader, desc=f'[{stage_name}] Epoch {epoch+1}/{num_epochs} [Train]'):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation Phase
        model.eval()
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f'[{stage_name}] Epoch {epoch+1}/{num_epochs} [Val]'):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
                
        val_accuracy = accuracy_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds, average='weighted')
        
        print(f"[{stage_name}] Epoch {epoch+1}/{num_epochs}")
        print(f"Train Loss: {epoch_loss:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}")
        
        # --- LOGIKA PENYIMPANAN MODEL TERBAIK ---
        if save_best and val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            torch.save(model.state_dict(), model_save_path)
            print(f"🌟 Model terbaik baru berhasil disimpan! (Val Acc meningkat menjadi {best_val_acc:.4f})")
            
        print("-" * 50)


# ===================================================================
# === 7. EKSEKUSI TRAINING 3 TAHAP (GRADUAL UNFREEZING) ===
# ===================================================================
print("\n" + "="*50)
print("MEMULAI PROSES PELATIHAN 3 TAHAP (GRADUAL UNFREEZING)")
print("="*50 + "\n")

# --- STAGE 1 ---
set_trainable_stage(model, stage=1)
optimizer_s1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_S1)
train_model(model, train_loader, val_loader, optimizer_s1, num_epochs=EPOCHS_S1, stage_name="STAGE 1")

# --- STAGE 2 ---
set_trainable_stage(model, stage=2)
optimizer_s2 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_S2)
train_model(model, train_loader, val_loader, optimizer_s2, num_epochs=EPOCHS_S2, stage_name="STAGE 2")

# --- STAGE 3 (DENGAN SAVE BEST MODEL) ---
set_trainable_stage(model, stage=3)
optimizer_s3 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_S3)
# Parameter save_best diaktifkan di sini
train_model(model, train_loader, val_loader, optimizer_s3, num_epochs=EPOCHS_S3, stage_name="STAGE 3", save_best=True, model_save_path="best_eva02_model.pth")


MEMULAI PROSES PELATIHAN 3 TAHAP (GRADUAL UNFREEZING)

Mempersiapkan Stage 1: Hanya Classification Head yang dilatih.


[STAGE 1] Epoch 1/3 [Train]:   0%|          | 0/65 [00:00<?, ?it/s]

[STAGE 1] Epoch 1/3 [Val]:   0%|          | 0/9 [00:00<?, ?it/s]

[STAGE 1] Epoch 1/3
Train Loss: 2.0106 | Val Acc: 0.6000 | Val F1: 0.5895
--------------------------------------------------


[STAGE 1] Epoch 2/3 [Train]:   0%|          | 0/65 [00:00<?, ?it/s]

[STAGE 1] Epoch 2/3 [Val]:   0%|          | 0/9 [00:00<?, ?it/s]

... (60 outputs left out)

[STAGE 3] Epoch 14/15
Train Loss: 0.0029 | Val Acc: 0.9846 | Val F1: 0.9844
--------------------------------------------------


[STAGE 3] Epoch 15/15 [Train]:   0%|          | 0/65 [00:00<?, ?it/s]

[STAGE 3] Epoch 15/15 [Val]:   0%|          | 0/9 [00:00<?, ?it/s]

[STAGE 3] Epoch 15/15
Train Loss: 0.0028 | Val Acc: 0.9846 | Val F1: 0.9844
--------------------------------------------------
